In [4]:
# === SESSION START: run this first, every time ===
from google.colab import drive, userdata
drive.mount('/content/drive')

REPO_DIR = "/content/drive/MyDrive/projects/namibia-baseline-pack-demo"
%cd "{REPO_DIR}"

import sys, importlib
sys.path.insert(0, f"{REPO_DIR}/src")

import ee, geemap
try:
    ee.Initialize(project=userdata.get("EE_PROJECT"))
except Exception:
    ee.Authenticate()
    ee.Initialize(project=userdata.get("EE_PROJECT"))
print("EE ready:", ee.String("ok").getInfo())

# --- state check: what is actually in the repo right now ---
!find . -path ./.git -prune -o -path ./outputs -prune -o -print | sort
!ls -lh outputs/rasters outputs/figures outputs/tables 2>/dev/null
!git status --short && git log --oneline -3

Mounted at /content/drive
/content/drive/MyDrive/projects/namibia-baseline-pack-demo
EE ready: ok
.
./config.yaml
./CREDITS.md
./data
./data/vector
./data/vector/aoi.gpkg
./data/vector/training.gpkg
./docs
./docs/class_key.md
./docs/time_log.md
./.gitignore
./LICENSE
./notebooks
./notebooks/05_rf_spectral.ipynb
./notebooks/.gitkeep
./README.md
./requirements.txt
./src
./src/baseline_pack
./src/baseline_pack/aoi.py
./src/baseline_pack/classify.py
./src/baseline_pack/composites.py
./src/baseline_pack/gitutils.py
./src/baseline_pack/__init__.py
./src/baseline_pack/__pycache__
./src/baseline_pack/__pycache__/aoi.cpython-313.pyc
./src/baseline_pack/__pycache__/classify.cpython-313.pyc
./src/baseline_pack/__pycache__/composites.cpython-313.pyc
./src/baseline_pack/__pycache__/gitutils.cpython-313.pyc
./src/baseline_pack/__pycache__/__init__.cpython-313.pyc
./src/baseline_pack/__pycache__/samples.cpython-313.pyc
./src/baseline_pack/__pycache__/trend.cpython-313.pyc
./src/baseline_pack/samples.

In [ ]:
# === SESSION 5 v2: everything in one cell (run only after the session-start cell) ===
import importlib, hashlib, inspect, time, glob, shutil, os, yaml
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.mask import mask
from pathlib import Path
import baseline_pack.classify as cl; importlib.reload(cl)
import baseline_pack.samples as sp; importlib.reload(sp)
import baseline_pack.gitutils as gu; importlib.reload(gu)
from baseline_pack.aoi import gdf_to_ee
from baseline_pack.composites import seasonal_composite, terrain

VERSION = "v2"
WEIGHTS = {1: 300, 2: 600, 3: 300, 4: 300}   # bush savanna (dominant class) gets double weight

# --- 0. safety checks: stop if the new inputs are not really there ---
poly = gpd.read_file("data/vector/training.gpkg", layer="training")
print(f"Training file: {len(poly)} polygons (104 before the targeted additions)")


assert len(poly) > 104, "STOP: new polygons not in the Drive file (upload with Replace, QGIS closed)"
assert poly["class_id"].notna().all(), "STOP: a polygon has no class_id"
assert "isinstance(max_per_class, dict)" in inspect.getsource(sp.balance), "STOP: samples.py balance() not saved"

# --- 1. settings, composites, features ---
cfg = yaml.safe_load(open("config.yaml"))
REMAP = {int(k): v for k, v in cfg["label_remap"].items()}
NAMES = {int(k): v["name"] for k, v in cfg["classes"].items()}
PALETTE = [cfg["classes"][k]["color"].lstrip("#") for k in sorted(cfg["classes"])]
CLASSES = sorted(NAMES)
aoi_gdf = gpd.read_file("data/vector/aoi.gpkg", layer="aoi")
aoi = gdf_to_ee(aoi_gdf).geometry()
region = gdf_to_ee(gpd.read_file("data/vector/aoi.gpkg", layer="aoi_buffer")).geometry()
dry, _, _ = seasonal_composite(region, cfg["current_year"], cfg["dry_season_months"], "dry")
wet, _, _ = seasonal_composite(region, cfg["current_year"], cfg["wet_season_months"], "wet")
stack = cl.feature_stack(dry, wet, terrain(region))
FEATURES = stack.bandNames().getInfo()
assert len(FEATURES) == 33, f"STOP: expected 33 features, got {len(FEATURES)}"

# --- 2. training pixels (new polygons -> new fingerprint -> new extraction) ---
train_gdf = cl.prepare_training(poly)
fp = hashlib.md5(open("data/vector/training.gpkg", "rb").read() + ",".join(FEATURES).encode()).hexdigest()[:8]
CSV = f"outputs/tables/training_pixels_2025_{fp}.csv"
if Path(CSV).exists():
    px = pd.read_csv(CSV); print("Loaded saved pixels:", CSV)
else:
    print("Extracting training pixels (background export)…")
    px = sp.extract_pixels_to_csv(stack, train_gdf, CSV, name=f"training_pixels_{fp}")
px = px.dropna(subset=FEATURES)
px["class_id"] = px["class_id"].map(REMAP)
train_gdf["class_id"] = train_gdf["class_id"].map(REMAP)
bal = sp.balance(px, max_per_class=WEIGHTS)
print("Pixels used by the model:"); display(bal.groupby(["class_id", "split"]).size().unstack())
balanced = sp.to_fc(bal, FEATURES)

# --- 3. diagnostic check on held-out polygons ---
clf_diag = cl.train_rf(balanced.filter(ee.Filter.eq("split", "train")), FEATURES)
cm, per_class, oa = cl.holdout_check(clf_diag, balanced.filter(ee.Filter.eq("split", "test")), CLASSES, NAMES)
print(f"Holdout overall accuracy (diagnostic only): {oa}%"); display(cm, per_class)

# --- 4. final model, classify, background export of the map ---
clf = cl.train_rf(balanced, FEATURES)
lc = stack.classify(clf).rename("class").toByte()
name = f"landcover_2025_rf_spectral_{VERSION}"
task = ee.batch.Export.image.toDrive(image=lc, description=name, folder="baseline_pack_exports",
                                     fileNamePrefix=name, region=region, scale=10,
                                     crs="EPSG:32733", maxPixels=1e9)
task.start()
while task.active():
    print("Map export running…", task.status()["state"]); time.sleep(30)
assert task.status()["state"] == "COMPLETED", task.status()
for _ in range(20):
    hits = glob.glob(f"/content/drive/MyDrive/baseline_pack_exports/{name}*.tif")
    if hits: break
    time.sleep(15)
out_tif = f"outputs/rasters/{name}.tif"
shutil.copy(max(hits, key=os.path.getmtime), out_tif); print("Saved:", out_tif)

# --- 5. mapped areas (pixel counting; final areas come in Session 9), compared with v1 ---
with rasterio.open(out_tif) as src:
    arr, _ = mask(src, aoi_gdf.geometry, crop=True, nodata=0)
    px_ha = abs(src.res[0] * src.res[1]) / 1e4
vals, counts = np.unique(arr[arr > 0], return_counts=True)
areas = pd.DataFrame({"class_name": [NAMES[int(v)] for v in vals],
                      f"{VERSION}_ha": (counts * px_ha).round(1)}, index=vals.astype(int))
areas[f"{VERSION}_%"] = (100 * areas[f"{VERSION}_ha"] / areas[f"{VERSION}_ha"].sum()).round(1)
areas.to_excel(f"outputs/tables/landcover_2025_{VERSION}_mapped_area.xlsx")
v1_file = "outputs/tables/landcover_2025_v1_mapped_area.xlsx"
if Path(v1_file).exists():
    areas["v1_%"] = pd.read_excel(v1_file, index_col=0)["percent"]
display(areas)

# --- 6. map to look at ---
Map = geemap.Map(basemap="HYBRID")
Map.addLayer(dry, {"bands": ["dry_B8", "dry_B4", "dry_B3"], "min": 0, "max": 0.35}, "Dry 2025 false colour", False)
Map.addLayer(wet, {"bands": ["wet_B8", "wet_B4", "wet_B3"], "min": 0, "max": 0.35}, "Wet 2025 false colour", False)
Map.addLayer(lc, {"min": 1, "max": 4, "palette": PALETTE}, f"Land cover {VERSION}")
Map.addLayer(gdf_to_ee(aoi_gdf).style(color="cyan", fillColor="00000000", width=2), {}, "AOI")
Map.add_legend(title=f"Land cover {VERSION}", keys=[NAMES[c] for c in CLASSES], colors=["#" + p for p in PALETTE])
Map.centerObject(aoi, 12); display(Map)

# --- 7. save to GitHub ---
gu.git_push(f"Session 5: land cover {VERSION} (targeted polygons, weighted bush savanna)")

Training file: 120 polygons (104 before the targeted additions)
Extracting training pixels (background export)…
Export running… READY
Pixels used by the model:


split,test,train
class_id,,
1,95,163
2,196,379
3,150,300
4,114,178


Holdout overall accuracy (diagnostic only): 67.9%


,Closed woody vegetation,Bush savanna,Open land (grass or bare),Built-up
Closed woody vegetation,84,11,0,0
Bush savanna,38,89,69,0
Open land (grass or bare),1,33,113,3
Built-up,0,0,23,91


,producer_acc_%,user_acc_%
Closed woody vegetation,88.4,68.3
Bush savanna,45.4,66.9
Open land (grass or bare),75.3,55.1
Built-up,79.8,96.8


Map export running… READY
Map export running… RUNNING
Map export running… RUNNING
Map export running… RUNNING
Map export running… RUNNING
Map export running… RUNNING
Map export running… RUNNING
Map export running… RUNNING
Map export running… RUNNING
Map export running… RUNNING
Map export running… RUNNING
Map export running… RUNNING
Map export running… RUNNING
Map export running… RUNNING
Map export running… RUNNING
Map export running… RUNNING
Map export running… RUNNING
Map export running… RUNNING


In [1]:
!git fetch origin && git status -sb

fatal: not a git repository (or any of the parent directories): .git


In [2]:
%cd /content/drive/MyDrive/projects/namibia-baseline-pack-demo
!pwd
!ls -a
!git fetch origin && git status -sb

[Errno 2] No such file or directory: '/content/drive/MyDrive/projects/namibia-baseline-pack-demo'
/content
/content
.  ..  .config	sample_data
fatal: not a git repository (or any of the parent directories): .git


In [5]:
!pwd
!ls -a
!git fetch origin && git status -sb

/content/drive/MyDrive/projects/namibia-baseline-pack-demo
.   config.yaml  data  .git	   LICENSE    outputs	 requirements.txt
..  CREDITS.md	 docs  .gitignore  notebooks  README.md  src
From https://github.com/ybrosig-geo/namibia-baseline-pack-demo
   c93b76a..83ddf9b  main       -> origin/main
## main...origin/main
 M data/vector/training.gpkg
 M notebooks/05_rf_spectral.ipynb
 M src/baseline_pack/samples.py


In [6]:
import baseline_pack.gitutils as gu; importlib.reload(gu)
gu.git_push("Session 5 closed")

git add → 
git commit → [main 68f6179] Session 5 closed
 3 files changed, 6 insertions(+), 5 deletions(-)
git push → warning: redirecting to https://github.com/ybrosig-geo/namibia-baseline-pack-demo.git/
To https://github.com/ybrosig-geo/namibia-baseline-pack-demo.git
   83ddf9b..68f6179  HEAD -> main
